#Databricks notebook: Bronze - Focos de Queimadas (Diário)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
import datetime
import re

spark = SparkSession.builder.getOrCreate()

In [0]:
# ================================================================
# Parâmetro recebido via Databricks Job
# ================================================================
dbutils.widgets.text("schema", "")
dbutils.widgets.text("table", "")
dbutils.widgets.text("path_raw", "")
dbutils.widgets.text("data_ref_carga", "")

schema = dbutils.widgets.get("schema")
table = dbutils.widgets.get("table")
path_raw = dbutils.widgets.get("path_raw")
data_ref_carga = dbutils.widgets.get("data_ref_carga")

if not data_ref_carga:
    raise ValueError("❌ Parâmetro 'data_ref_carga' não informado")

print(f"📅 Data de referência: {data_ref_carga}")
print(f"📂 Buscando arquivos em: {path_raw}")

# ==========================
# 🔹 Listar arquivos do RAW
# ==========================
files = [
    f.path for f in dbutils.fs.ls(path_raw)
    if f.name.lower().startswith("focos_diario_br_") and f.name.lower().endswith(".csv")
]

if not files:
    raise FileNotFoundError("⚠️ Nenhum arquivo focos_diario_br encontrado no RAW")

print("📄 Arquivos encontrados:", files)

In [0]:
# ==========================
# 🔹 Função para extrair data do nome
# ==========================
def extrair_data(nome):
    match = re.search(r"(\d{8})", nome)
    if match:
        return datetime.strptime(match.group(1), "%Y%m%d")
    return datetime.min

# ==========================
# 🔹 Ordenar e pegar o mais recente
# ==========================
files_sorted = sorted(
    files,
    key=lambda p: extrair_data(p.split("/")[-1]),
    reverse=True
)

In [0]:
file_path = files_sorted[0]
file_name = file_path.split("/")[-1]

print(f"✅ Arquivo selecionado: {file_name}")

In [0]:
# ==========================
# 🔹 Leitura do CSV
# ==========================
df = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(file_path)
)

df = df.withColumn("data_ref_carga", lit(data_ref_carga))

print(f"✅ Linhas lidas: {df.count()}")

In [0]:
# ==========================
# 🔹 Escrita na Tabela Bronze
# ==========================
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"data_ref_carga = '{data_ref_carga}'")
    .partitionBy("data_ref_carga")
    .saveAsTable(f"{schema}.{table}")
)

print(f"💾 Dados gravados em: {schema}.{table}")